In [1]:
# Environment Setup & Core Dependencies
import numpy as np
import pandas as pd

# Pandas display configuration for analytical clarity
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("Dependencies loaded successfully.")

Dependencies loaded successfully.


In [2]:
# Data Ingestion & Initial Schema Inspection
# Loading the primary dataset
df = pd.read_csv('Dataset.csv', sep='\t')

print(f"Initial Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns\n")
print("First 3 Records:")
display(df.head(3))

print("\nMissing Values Audit:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Initial Dataset Dimensions: 2240 rows, 29 columns

First 3 Records:


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,NumDealsPurchases,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.00,0,0,04-09-2012,58,635,88,546,172,88,88,3,8,10,4,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.00,1,1,08-03-2014,38,11,1,6,2,1,6,2,1,1,2,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.00,0,0,21-08-2013,26,426,49,127,111,21,42,1,8,2,10,4,0,0,0,0,0,0,3,11,0



Missing Values Audit:
Income    24
dtype: int64


In [3]:
# Feature Sanitation & Redundancy Elimination
# Dropping system-level invariant variables ('Z_CostContact' and 'Z_Revenue' contain identical values across all observations)
invariant_columns = ['Z_CostContact', 'Z_Revenue']
df.drop(columns=invariant_columns, inplace=True, errors='ignore')

print(f"Removed invariant features: {invariant_columns}")
print(f"Current Schema Dimensions: {df.shape}")

Removed invariant features: ['Z_CostContact', 'Z_Revenue']
Current Schema Dimensions: (2240, 27)


In [4]:
# Context-Aware Missing Value Imputation
# Imputing missing 'Income' values using median values aggregated by Education level and Marital Status
income_medians = df.groupby(['Education', 'Marital_Status'])['Income'].transform('median')
df['Income'] = df['Income'].fillna(income_medians)

# Verification
print(f"Remaining null values in Income: {df['Income'].isnull().sum()}")

Remaining null values in Income: 0


In [5]:
# Outlier Remediation & Logical Validation
# Filtering biologically implausible birth years (e.g., birth year < 1900)
valid_age_condition = df['Year_Birth'] >= 1900

# Filtering extreme income outliers (e.g., Income > $600,000)
valid_income_condition = df['Income'] < 600000

# Applying logical filters
df_clean = df[valid_age_condition & valid_income_condition].copy()

print(f"Records dropped during validation: {len(df) - len(df_clean)}")
print(f"Cleaned Dataset Dimensions: {df_clean.shape}")

Records dropped during validation: 3
Cleaned Dataset Dimensions: (2237, 27)


In [6]:
# Domain-Specific Feature Construction
# Setting temporal baseline reference
CURRENT_YEAR = 2026

# 1. Demographic Indicators
df_clean['Customer_Age'] = CURRENT_YEAR - df_clean['Year_Birth']
df_clean['Total_Children'] = df_clean['Kidhome'] + df_clean['Teenhome']
df_clean['Is_Parent'] = (df_clean['Total_Children'] > 0).astype(int)

# 2. Monetary Aggregations
spending_fields = ['MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']
df_clean['Total_Spending'] = df_clean[spending_fields].sum(axis=1)

# 3. Behavioral Channel Metrics
purchase_fields = ['NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases']
df_clean['Total_Purchases'] = df_clean[purchase_fields].sum(axis=1)

# 4. Historical Marketing Engagement
campaign_fields = ['AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5']
df_clean['Total_Accepted_Cmp'] = df_clean[campaign_fields].sum(axis=1)

# 5. Customer Tenure (Days from registration relative to baseline date)
df_clean['Dt_Customer'] = pd.to_datetime(df_clean['Dt_Customer'], format='%d-%m-%Y')
REFERENCE_DATE = pd.to_datetime('2026-09-24')
df_clean['Enrollment_Days'] = (REFERENCE_DATE - df_clean['Dt_Customer']).dt.days

print("Feature engineering successfully executed.")
display(df_clean[['Customer_Age', 'Total_Children', 'Total_Spending', 'Total_Purchases', 'Total_Accepted_Cmp', 'Enrollment_Days']].describe())

Feature engineering successfully executed.


,Customer_Age,Total_Children,Total_Spending,Total_Purchases,Total_Accepted_Cmp,Enrollment_Days
count,2237.00,2237.00,2237.00,2237.00,2237.00,2237.00
mean,57.13,0.95,605.74,12.54,0.30,4823.74
std,11.79,0.75,601.84,7.21,0.68,202.14
min,30.00,0.00,5.00,0.00,0.00,4470.00
25%,49.00,0.00,69.00,6.00,0.00,4651.00
50%,56.00,1.00,396.00,12.00,0.00,4826.00
75%,67.00,1.00,1045.00,18.00,0.00,4999.00
max,126.00,3.00,2525.00,32.00,4.00,5169.00


In [7]:
# Persisting Processed Data
df_clean.to_csv('cleaned_customer_data.csv', index=False)
print("Cleaned dataset saved as 'cleaned_customer_data.csv'.")

Cleaned dataset saved as 'cleaned_customer_data.csv'.
